# Rozdział 7: Tworzenie aplikacji czatowych
## Szybki start z OpenAI API

Ten notatnik jest adaptacją [repozytorium przykładów Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), które zawiera notatniki uzyskujące dostęp do usług [Azure OpenAI](notebook-azure-openai.ipynb).

Pythonowe OpenAI API działa również z modelami Azure OpenAI, z kilkoma modyfikacjami. Dowiedz się więcej o różnicach tutaj: [Jak przełączać się między punktami końcowymi OpenAI i Azure OpenAI w Pythonie](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Przegląd  
"Duże modele językowe to funkcje mapujące tekst na tekst. Dla podanego ciągu wejściowego tekstu, duży model językowy próbuje przewidzieć, jaki tekst pojawi się następny"(1). Ten notatnik „quickstart” wprowadzi użytkowników w podstawowe pojęcia dotyczące LLM, kluczowe wymagania pakietu do rozpoczęcia pracy z AML, łagodne wprowadzenie do projektowania promptów oraz kilka krótkich przykładów różnych zastosowań. 


## Spis treści  

[Przegląd](#overview)  
[Jak korzystać z usługi OpenAI](#how-to-use-openai-service)  
[1. Tworzenie Twojej usługi OpenAI](#1.-creating-your-openai-service)  
[2. Instalacja](#2.-installation)    
[3. Dane uwierzytelniające](#3.-credentials)  

[Przypadki użycia](#use-cases)    
[1. Streszczanie tekstu](#1.-summarize-text)  
[2. Klasyfikacja tekstu](#2.-classify-text)  
[3. Generowanie nowych nazw produktów](#3.-generate-new-product-names)  
[4. Dostrajanowanie klasyfikatora](#4.fine-tune-a-classifier)  

[Bibliografia](#references)


### Zbuduj swój pierwszy prompt  
To krótkie ćwiczenie zapewni podstawowe wprowadzenie do przesyłania promptów do modelu OpenAI dla prostego zadania "podsumowanie".


**Kroki**:  
1. Zainstaluj bibliotekę OpenAI w swoim środowisku Pythona  
2. Załaduj standardowe biblioteki pomocnicze i ustaw typowe dane uwierzytelniające do usługi OpenAI, którą utworzyłeś  
3. Wybierz model do swojego zadania  
4. Stwórz prosty prompt dla modelu  
5. Prześlij swoje żądanie do API modelu!


### 1. Zainstaluj OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importuj pomocnicze biblioteki i zainicjuj poświadczenia


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Znajdowanie odpowiedniego modelu  
Modele takie jak GPT-4o i GPT-4o mini potrafią rozumieć i generować język naturalny oraz są dostępne na platformie OpenAI w różnych wariantach pod względem mocy i szybkości, odpowiednich do różnych zadań.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Projektowanie promptów  

"Magia dużych modeli językowych polega na tym, że poprzez trenowanie w celu minimalizacji błędu predykcji na ogromnych ilościach tekstu, modele uczą się konceptów przydatnych do tych przewidywań. Na przykład, uczą się konceptów takich jak"(1):

* jak pisać poprawnie ortograficznie
* jak działa gramatyka
* jak parafrazować
* jak odpowiadać na pytania
* jak prowadzić rozmowę
* jak pisać w wielu językach
* jak programować
* itp.

#### Jak kontrolować duży model językowy  
"Spośród wszystkich danych wejściowych do dużego modelu językowego, zdecydowanie najbardziej wpływowy jest tekst promptu(1).

Duże modele językowe można wywołać do generowania wyników na kilka sposobów:

Instrukcja: Powiedz modelowi, czego chcesz
Uzupełnienie: Nakłoń model do uzupełnienia początku tego, czego chcesz
Demonstracja: Pokaż modelowi, czego chcesz, za pomocą:
Kilku przykładów w promptcie
Setek lub tysięcy przykładów w zbiorze treningowym do fine-tuningu"



#### Istnieją trzy podstawowe wskazówki dotyczące tworzenia promptów:

**Pokaż i powiedz**. Wyraźnie określ, czego chcesz poprzez instrukcje, przykłady lub ich kombinację. Jeśli chcesz, aby model uszeregował listę elementów alfabetycznie lub sklasyfikował akapit według sentymentu, pokaż mu, że tego oczekujesz.

**Zapewnij wysokiej jakości dane**. Jeśli próbujesz zbudować klasyfikator lub zmusić model do podążania za wzorcem, upewnij się, że masz wystarczająco dużo przykładów. Dokładnie sprawdź swoje przykłady — model zwykle na tyle jest inteligentny, by przejrzeć podstawowe błędy ortograficzne i udzielić odpowiedzi, ale może też założyć, że jest to celowe i to wpłynąć na odpowiedź.

**Sprawdź ustawienia.** Parametry temperature i top_p kontrolują, jak deterministyczny jest model podczas generowania odpowiedzi. Jeśli oczekujesz odpowiedzi, gdzie jest tylko jedna właściwa odpowiedź, ustaw je niżej. Jeśli chcesz bardziej zróżnicowanych odpowiedzi, ustaw je wyżej. Najczęstszym błędem jest mylenie tych ustawień z suwakami „bystrości” czy „kreatywności”.


Źródło: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Prześlij!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Powtórz to samo wywołanie, jak porównują się wyniki?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Podsumuj tekst  
#### Wyzwanie  
Podsumuj tekst, dodając „tl;dr:” na końcu fragmentu tekstu. Zauważ, jak model rozumie, jak wykonać wiele zadań bez dodatkowych instrukcji. Możesz eksperymentować z bardziej opisowymi podpowiedziami niż tl;dr, aby zmodyfikować zachowanie modelu i dostosować otrzymywane podsumowanie(3).  

Ostatnie badania wykazały znaczne postępy w wielu zadaniach i benchmarkach NLP poprzez wstępne trenowanie na dużym korpusie tekstów, a następnie dostrajanie na określonym zadaniu. Choć architektura zwykle jest niezależna od zadania, ta metoda nadal wymaga specyficznych dla zadania zestawów danych do dostrajania, liczących tysiące lub dziesiątki tysięcy przykładów. Dla porównania, ludzie zwykle potrafią wykonać nowe zadanie językowe na podstawie zaledwie kilku przykładów lub prostych instrukcji – coś, z czym obecne systemy NLP nadal mają znaczne trudności. Tutaj pokazujemy, że skalowanie modeli językowych znacznie poprawia uniwersalne, niskoprzykładowe wyniki, czasem nawet osiągając konkurencyjność wobec wcześniejszych, najlepszych metod dostrajania. 



Tl;dr


# Ćwiczenia dla różnych przypadków użycia  
1. Podsumuj tekst  
2. Klasyfikuj tekst  
3. Generuj nowe nazwy produktów


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasyfikuj tekst  
#### Zadanie  
Klasyfikuj elementy do kategorii podanych podczas wywołania. W poniższym przykładzie podajemy zarówno kategorie, jak i tekst do sklasyfikowania w podpowiedzi (*playground_reference). 

Zapytanie klienta: Dzień dobry, ostatnio zepsuł się jeden z klawiszy na klawiaturze mojego laptopa i potrzebuję wymiany:

Sklasyfikowana kategoria:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Wygeneruj nowe nazwy produktów
#### Wyzwanie
Stwórz nazwy produktów na podstawie przykładowych słów. W promptcie zamieszczamy informacje o produkcie, dla którego zamierzamy wygenerować nazwy. Podajemy także podobny przykład, aby pokazać schemat, jaki chcemy otrzymać. Ustawiliśmy również wysoką wartość temperatury, aby zwiększyć losowość i uzyskać bardziej innowacyjne odpowiedzi.

Opis produktu: Domowy mikser do shake'ów mlecznych
Słowa klucze: szybki, zdrowy, kompaktowy.
Nazwy produktów: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis produktu: Para butów pasujących na każdą stopę.
Słowa klucze: adaptacyjny, dopasowany, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Odniesienia  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najlepsze praktyki dostrajania modeli GPT do klasyfikacji tekstu](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Po więcej pomocy  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Współpracownicy
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
